# Lotofácil — Histórico completo, rede neural e sugestão para o próximo concurso

Notebook **autocontido**: todas as funções estão embutidas nas células abaixo (mesma lógica de `lotofacil_analise.py`), então basta este arquivo `.ipynb` para rodar — não depende de nenhum outro arquivo do repositório.

O que ele faz:

1. Baixa automaticamente **todo o histórico** de concursos da Lotofácil (com cache local incremental em `lotofacil_historico.csv`, salvo na mesma pasta deste notebook);
2. Deixa você escolher **quantos concursos anteriores considerar** na análise (ex.: só os últimos 100);
3. Faz uma análise estatística do histórico (frequência, atraso, pares, paridade, soma, repetição, distribuição por coluna da cartela);
4. Treina uma **rede neural (MLP)** que aprende a associar a frequência recente de cada dezena à probabilidade dela sair no próximo concurso;
5. Mostra, para cada estratégia (combinada, mais quentes, tendência recente, atrasados e rede neural), sugestões com **15, 16 e 17 números**.

> **Importante:** a Lotofácil é um sorteio aleatório — cada concurso é um evento independente e concursos passados **não** influenciam o próximo sorteio. Isso vale também para a rede neural: ela é um modelo estatístico de verdade, treinado com scikit-learn, mas **não existe padrão real para ela aprender** em um sorteio uniforme e aleatório — o resultado dela não tem motivo para superar o acaso. Este notebook tem fins exploratórios/educacionais. Jogue com responsabilidade.


## 0. Configuração e funções auxiliares

Estas células definem tudo que o notebook usa: bibliotecas, download/cache do histórico, análise estatística, geração de sugestões e a rede neural. Só é preciso rodá-las uma vez, na ordem.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

import csv
import statistics
import time
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from itertools import combinations
from pathlib import Path

import requests

try:
    import numpy as np
    from sklearn.neural_network import MLPClassifier
    SKLEARN_DISPONIVEL = True
except ImportError:
    SKLEARN_DISPONIVEL = False

API_BASE = "https://servicebus2.caixa.gov.br/portaldeloterias/api/lotofacil"
FALLBACK_BULK_URL = "https://loteriascaixa-api.herokuapp.com/api/lotofacil"
CACHE_FILE = Path("lotofacil_historico.csv")  # salvo na mesma pasta do notebook
HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; LotofacilAnalise/1.0)"}

TOTAL_DEZENAS = 25
DEZENAS_POR_JOGO = 15  # dezenas sorteadas em cada concurso
MIN_DEZENAS_APOSTA = 15
MAX_DEZENAS_APOSTA = 20  # a Lotofácil aceita apostas de 15 a 20 números
QUANTIDADES_PADRAO = (15, 16, 17)  # tamanhos de aposta mostrados por categoria
JANELA_NN_PADRAO = 10  # concursos usados como "memória" da rede neural

# Layout da cartela oficial da Lotofácil (5 colunas x 5 linhas)
COLUNAS_CARTELA = {
    1: [1, 6, 11, 16, 21],
    2: [2, 7, 12, 17, 22],
    3: [3, 8, 13, 18, 23],
    4: [4, 9, 14, 19, 24],
    5: [5, 10, 15, 20, 25],
}



### Download e cache do histórico

In [ ]:
def buscar_concurso(session, numero=None, retries=3, timeout=10):
    """Busca um concurso específico (ou o mais recente, se numero=None)."""
    url = API_BASE if numero is None else f"{API_BASE}/{numero}"
    ultimo_erro = None
    for tentativa in range(1, retries + 1):
        try:
            resp = session.get(url, headers=HEADERS, timeout=timeout)
            resp.raise_for_status()
            return resp.json()
        except (requests.RequestException, ValueError) as exc:
            ultimo_erro = exc
            if tentativa < retries:
                time.sleep(1.5 * tentativa)
    raise ultimo_erro


def descobrir_ultimo_concurso(session):
    dados = buscar_concurso(session)
    return int(dados["numero"])


def parse_concurso(dados):
    numero = int(dados["numero"])
    data = dados.get("dataApuracao", "")
    dezenas = sorted(int(d) for d in dados["listaDezenas"])
    if len(dezenas) != DEZENAS_POR_JOGO:
        raise ValueError(f"Concurso {numero} com número inesperado de dezenas: {dezenas}")
    return numero, data, dezenas


def baixar_fallback_bulk(session, timeout=30):
    """Fonte alternativa: baixa todo o histórico em uma única requisição."""
    resp = session.get(FALLBACK_BULK_URL, headers=HEADERS, timeout=timeout)
    resp.raise_for_status()
    dados = resp.json()
    historico = {}
    for item in dados:
        numero = int(item["concurso"])
        data = item.get("data", "")
        dezenas = sorted(int(d) for d in item["dezenas"])
        historico[numero] = (data, dezenas)
    return historico


def carregar_cache():
    historico = {}
    if CACHE_FILE.exists():
        with CACHE_FILE.open(newline="", encoding="utf-8") as f:
            reader = csv.reader(f)
            next(reader, None)  # cabeçalho
            for linha in reader:
                if not linha:
                    continue
                numero = int(linha[0])
                data = linha[1]
                dezenas = sorted(int(x) for x in linha[2:2 + DEZENAS_POR_JOGO])
                historico[numero] = (data, dezenas)
    return historico


def salvar_cache(historico):
    with CACHE_FILE.open("w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["concurso", "data"] + [f"dezena_{i + 1}" for i in range(DEZENAS_POR_JOGO)])
        for numero in sorted(historico):
            data, dezenas = historico[numero]
            writer.writerow([numero, data] + dezenas)


def atualizar_historico(max_workers=12, forcar=False, apenas_ultimos=None):
    """Garante que o cache local tenha todo o histórico (ou os N mais recentes)."""
    historico = {} if forcar else carregar_cache()

    with requests.Session() as session:
        try:
            ultimo = descobrir_ultimo_concurso(session)
        except Exception as exc:
            print(f"Não foi possível acessar a API oficial da Caixa ({exc}).")
            print("Tentando fonte alternativa (API pública com histórico completo)...")
            historico_fallback = baixar_fallback_bulk(session)
            historico.update(historico_fallback)
            salvar_cache(historico)
            print(f"Histórico obtido via fonte alternativa: {len(historico)} concursos.")
            return historico

        primeiro_desejado = 1
        if apenas_ultimos:
            primeiro_desejado = max(1, ultimo - apenas_ultimos + 1)

        faltantes = [n for n in range(primeiro_desejado, ultimo + 1) if n not in historico]

        if not faltantes:
            print(f"Histórico já atualizado: {len(historico)} concursos em cache (último: {ultimo}).")
            return historico

        print(f"Baixando {len(faltantes)} concurso(s) novo(s) (até o concurso {ultimo})...")

        falhas = []
        concluidos = 0
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futuros = {executor.submit(buscar_concurso, session, n): n for n in faltantes}
            for futuro in as_completed(futuros):
                numero_alvo = futuros[futuro]
                concluidos += 1
                try:
                    dados = futuro.result()
                    numero, data, dezenas = parse_concurso(dados)
                    historico[numero] = (data, dezenas)
                except Exception:
                    falhas.append(numero_alvo)

                if concluidos % 200 == 0 or concluidos == len(faltantes):
                    print(f"  ... {concluidos}/{len(faltantes)} baixados")

        if falhas:
            amostra = sorted(falhas)[:10]
            sufixo = "..." if len(falhas) > 10 else ""
            print(f"Aviso: {len(falhas)} concurso(s) não puderam ser baixados: {amostra}{sufixo}")

        salvar_cache(historico)
        print(f"Histórico salvo em '{CACHE_FILE.name}': {len(historico)} concursos.")

    return historico


def historico_ordenado(historico, apenas_ultimos=None):
    numeros = sorted(historico)
    if apenas_ultimos:
        numeros = numeros[-apenas_ultimos:]
    return [(n, historico[n][0], historico[n][1]) for n in numeros]



### Análise estatística

In [ ]:
def analisar(lista_concursos, janela_recente=25):
    todos_jogos = [dezenas for _, _, dezenas in lista_concursos]
    total_concursos = len(todos_jogos)

    freq_total = Counter()
    for jogo in todos_jogos:
        freq_total.update(jogo)

    recentes = todos_jogos[-janela_recente:] if total_concursos >= janela_recente else todos_jogos
    freq_recente = Counter()
    for jogo in recentes:
        freq_recente.update(jogo)

    atraso = {}
    for dezena in range(1, TOTAL_DEZENAS + 1):
        contador = 0
        for jogo in reversed(todos_jogos):
            if dezena in jogo:
                atraso[dezena] = contador
                break
            contador += 1
        else:
            atraso[dezena] = total_concursos

    pares = Counter()
    for jogo in todos_jogos:
        pares.update(combinations(sorted(jogo), 2))

    contagem_pares_impares = Counter()
    for jogo in todos_jogos:
        n_pares = sum(1 for d in jogo if d % 2 == 0)
        contagem_pares_impares[n_pares] += 1

    somas = [sum(jogo) for jogo in todos_jogos]
    soma_media = statistics.mean(somas) if somas else 0
    soma_desvio = statistics.pstdev(somas) if len(somas) > 1 else 0

    repeticoes = [
        len(set(anterior) & set(atual))
        for anterior, atual in zip(todos_jogos, todos_jogos[1:])
    ]
    repeticao_media = statistics.mean(repeticoes) if repeticoes else 0

    freq_coluna = Counter()
    for jogo in todos_jogos:
        for dezena in jogo:
            for coluna, dezenas_coluna in COLUNAS_CARTELA.items():
                if dezena in dezenas_coluna:
                    freq_coluna[coluna] += 1
                    break

    return {
        "total_concursos": total_concursos,
        "janela_recente": janela_recente,
        "freq_total": freq_total,
        "freq_recente": freq_recente,
        "atraso": atraso,
        "pares": pares,
        "contagem_pares_impares": contagem_pares_impares,
        "somas": somas,
        "soma_media": soma_media,
        "soma_desvio": soma_desvio,
        "repeticao_media": repeticao_media,
        "freq_coluna": freq_coluna,
    }



### Geração da sugestão de jogo (por estratégia e por tamanho de aposta)

In [ ]:
def _sugestao_combinada(resultado, pesos=(0.35, 0.45, 0.20), quantidade=DEZENAS_POR_JOGO):
    """Combina frequência histórica, tendência recente e atraso em um score
    por dezena. Monta um núcleo de 15 dezenas ajustado pela soma histórica
    e, se a aposta pedida tiver mais de 15 números (desdobramento de 16 a
    20 dezenas, como a Lotofácil permite), completa com as próximas
    melhores dezenas do ranking."""
    freq_total = resultado["freq_total"]
    freq_recente = resultado["freq_recente"]
    atraso = resultado["atraso"]

    max_freq_total = max(freq_total.values()) if freq_total else 1
    max_freq_recente = max(freq_recente.values()) if freq_recente else 1
    max_atraso = max(atraso.values()) if atraso else 1

    w_total, w_recente, w_atraso = pesos
    pontuacao = {}
    for dezena in range(1, TOTAL_DEZENAS + 1):
        score = (
            w_total * (freq_total.get(dezena, 0) / max_freq_total)
            + w_recente * (freq_recente.get(dezena, 0) / max_freq_recente)
            + w_atraso * (atraso.get(dezena, 0) / max_atraso)
        )
        pontuacao[dezena] = score

    ranking = sorted(pontuacao, key=lambda d: pontuacao[d], reverse=True)

    nucleo = _ajustar_soma(
        ranking[:DEZENAS_POR_JOGO], ranking, resultado["soma_media"], resultado["soma_desvio"]
    )

    if quantidade > DEZENAS_POR_JOGO:
        extras = [d for d in ranking if d not in nucleo][: quantidade - DEZENAS_POR_JOGO]
        sugestao = nucleo + extras
    else:
        sugestao = nucleo[:quantidade]

    return sorted(sugestao), pontuacao


def _ajustar_soma(sugestao, ranking, media_soma, desvio_soma, max_iter=40):
    """Troca dezenas da seleção por reservas sempre que a soma total sair da
    faixa média +/- 1 desvio padrão do histórico. A cada iteração, entre
    todas as trocas (membro da seleção <-> reserva) que aproximam a soma da
    média, prioriza-se remover o membro pior rankeado. Uma troca só é
    aplicada se reduzir estritamente a distância até a média, o que garante
    que o processo sempre termina (não há como oscilar indefinidamente)."""
    if desvio_soma <= 0:
        return list(sugestao)

    faixa_min, faixa_max = media_soma - desvio_soma, media_soma + desvio_soma
    sugestao = list(sugestao)
    reserva = [d for d in ranking if d not in sugestao]

    for _ in range(max_iter):
        soma_atual = sum(sugestao)
        if faixa_min <= soma_atual <= faixa_max:
            break

        diferenca_atual = abs(soma_atual - media_soma)
        melhor_swap = None

        # avalia os membros do pior para o melhor rankeado, aceitando a
        # primeira troca que melhore a distância até a média
        for membro in sorted(sugestao, key=lambda d: -ranking.index(d)):
            candidatos_bons = [
                c for c in reserva
                if abs((soma_atual - membro + c) - media_soma) < diferenca_atual
            ]
            if candidatos_bons:
                melhor_candidato = min(
                    candidatos_bons,
                    key=lambda c: abs((soma_atual - membro + c) - media_soma),
                )
                melhor_swap = (membro, melhor_candidato)
                break

        if melhor_swap is None:
            break  # nenhuma troca melhora a soma; evita loop sem progresso

        membro, candidato = melhor_swap
        sugestao.remove(membro)
        sugestao.append(candidato)
        reserva.remove(candidato)
        reserva.append(membro)

    return sugestao


def gerar_sugestoes(resultado, quantidades=QUANTIDADES_PADRAO, pontuacao_nn=None):
    """Para cada estratégia (combinada, mais quentes, tendência recente,
    atrasados e, se disponível, rede neural), monta uma sugestão para cada
    tamanho de aposta em `quantidades` (ex.: 15, 16 e 17 números).

    Retorna um dict {nome_da_estrategia: {quantidade: [dezenas]}}.
    """
    ranking_total = [d for d, _ in resultado["freq_total"].most_common(TOTAL_DEZENAS)]
    ranking_recente = [d for d, _ in resultado["freq_recente"].most_common(TOTAL_DEZENAS)]
    ranking_atraso = sorted(
        range(1, TOTAL_DEZENAS + 1), key=lambda d: -resultado["atraso"].get(d, 0)
    )

    sugestoes = {"Combinada (frequência + tendência + atraso)": {}}
    for quantidade in quantidades:
        jogo, _pontuacao = _sugestao_combinada(resultado, quantidade=quantidade)
        sugestoes["Combinada (frequência + tendência + atraso)"][quantidade] = jogo

    rankings_simples = {
        "Mais quentes (frequência histórica)": ranking_total,
        "Tendência recente": ranking_recente,
        "Números atrasados": ranking_atraso,
    }
    if pontuacao_nn is not None:
        rankings_simples["Rede neural (MLP)"] = sorted(
            pontuacao_nn, key=lambda d: pontuacao_nn[d], reverse=True
        )

    for nome, ranking in rankings_simples.items():
        sugestoes[nome] = {q: sorted(ranking[:q]) for q in quantidades}

    return sugestoes



### Rede neural (MLP)

Um palpite "baseado em rede neural" foi pedido explicitamente, e é importante ser honesto sobre o que ele é: a Lotofácil é um sorteio aleatório e uniforme, sem padrão real para uma rede aprender. O modelo abaixo é uma **rede neural de verdade** — um MLP (Multi-Layer Perceptron, perceptron multicamadas) treinado com `scikit-learn` — e o treino/inferência são legítimos, mas o resultado não tem nenhum motivo estatístico para superar o acaso.

Como funciona: para cada concurso, a entrada é a frequência de cada dezena nos últimos `janela` concursos (normalizada), e o rótulo é o vetor binário de 25 posições indicando quais dezenas saíram — um problema de classificação **multirrótulo**. Depois de treinada, a rede prevê a probabilidade de cada dezena sair no próximo concurso a partir da janela mais recente.

In [ ]:
def _construir_dataset_nn(todos_jogos, janela):
    """Para cada concurso a partir do (janela+1)-ésimo, a entrada é a
    frequência de cada dezena nos `janela` concursos anteriores (normalizada
    entre 0 e 1) e o rótulo é o vetor binário de 25 posições indicando quais
    dezenas saíram naquele concurso (classificação multirrótulo)."""
    X, y = [], []
    for i in range(janela, len(todos_jogos)):
        contagem = Counter()
        for jogo in todos_jogos[i - janela:i]:
            contagem.update(jogo)
        X.append([contagem.get(d, 0) / janela for d in range(1, TOTAL_DEZENAS + 1)])
        y.append([1 if d in todos_jogos[i] else 0 for d in range(1, TOTAL_DEZENAS + 1)])
    return np.array(X), np.array(y)


def treinar_rede_neural(lista_concursos, janela=JANELA_NN_PADRAO, random_state=42):
    """Treina um MLPClassifier multirrótulo e retorna a probabilidade
    prevista de cada dezena (1 a 25) aparecer no próximo concurso."""
    if not SKLEARN_DISPONIVEL:
        raise RuntimeError(
            "scikit-learn não está instalado. Rode: pip install scikit-learn"
        )

    todos_jogos = [dezenas for _, _, dezenas in lista_concursos]
    minimo_necessario = janela + 20
    if len(todos_jogos) < minimo_necessario:
        raise ValueError(
            f"Histórico insuficiente para treinar a rede neural: "
            f"são necessários pelo menos {minimo_necessario} concursos "
            f"(há {len(todos_jogos)}). Aumente --apenas-ultimos/CONCURSOS_A_CONSIDERAR "
            "ou diminua a janela da rede neural."
        )

    X, y = _construir_dataset_nn(todos_jogos, janela)

    modelo = MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation="relu",
        max_iter=2000,
        random_state=random_state,
        early_stopping=True,
    )
    modelo.fit(X, y)

    contagem_atual = Counter()
    for jogo in todos_jogos[-janela:]:
        contagem_atual.update(jogo)
    vetor_atual = np.array(
        [[contagem_atual.get(d, 0) / janela for d in range(1, TOTAL_DEZENAS + 1)]]
    )

    probabilidades = modelo.predict_proba(vetor_atual)[0]
    return {d: float(p) for d, p in zip(range(1, TOTAL_DEZENAS + 1), probabilidades)}



### Relatório em texto

In [ ]:
def montar_relatorio(
    resultado, lista_concursos, quantidades=QUANTIDADES_PADRAO, pontuacao_nn=None
):
    total = resultado["total_concursos"]
    janela_recente = resultado["janela_recente"]
    primeiro, ultimo = lista_concursos[0][0], lista_concursos[-1][0]
    data_ultimo = lista_concursos[-1][1]

    linhas = []
    linhas.append(
        f"\nConcursos analisados: {total} (do {primeiro} ao {ultimo}, "
        f"último sorteio em {data_ultimo})"
    )

    linhas.append("\n--- NÚMEROS MAIS FREQUENTES (histórico completo) ---")
    for dezena, qtd in resultado["freq_total"].most_common(10):
        linhas.append(f"  {dezena:02d} -> saiu {qtd} vezes ({100 * qtd / total:.1f}%)")

    linhas.append("\n--- NÚMEROS MENOS FREQUENTES (histórico completo) ---")
    for dezena, qtd in sorted(resultado["freq_total"].items(), key=lambda x: x[1])[:10]:
        linhas.append(f"  {dezena:02d} -> saiu {qtd} vezes ({100 * qtd / total:.1f}%)")

    linhas.append(f"\n--- TENDÊNCIA RECENTE (últimos {janela_recente} concursos) ---")
    for dezena, qtd in resultado["freq_recente"].most_common(10):
        linhas.append(f"  {dezena:02d} -> saiu {qtd}x nos últimos {janela_recente}")

    linhas.append("\n--- NÚMEROS MAIS ATRASADOS (há mais tempo sem sair) ---")
    for dezena, atraso in sorted(resultado["atraso"].items(), key=lambda x: -x[1])[:10]:
        linhas.append(f"  {dezena:02d} -> {atraso} concurso(s) sem sair")

    linhas.append("\n--- PARES DE NÚMEROS QUE MAIS SAEM JUNTOS ---")
    for (a, b), qtd in resultado["pares"].most_common(10):
        linhas.append(f"  {a:02d} e {b:02d} -> juntos em {qtd} concursos")

    linhas.append("\n--- DISTRIBUIÇÃO PARES x ÍMPARES POR JOGO ---")
    for n_pares, qtd in sorted(resultado["contagem_pares_impares"].items()):
        linhas.append(f"  {n_pares} pares / {15 - n_pares} ímpares -> {qtd} concurso(s)")

    linhas.append("\n--- SOMA DAS 15 DEZENAS SORTEADAS ---")
    linhas.append(
        f"  Média histórica: {resultado['soma_media']:.1f}  |  "
        f"Desvio padrão: {resultado['soma_desvio']:.1f}"
    )

    linhas.append("\n--- REPETIÇÃO EM RELAÇÃO AO CONCURSO ANTERIOR ---")
    linhas.append(
        f"  Em média, {resultado['repeticao_media']:.1f} dezenas se repetem "
        "de um concurso para o seguinte."
    )

    linhas.append("\n--- DISTRIBUIÇÃO POR COLUNA DA CARTELA OFICIAL ---")
    for coluna, qtd in sorted(resultado["freq_coluna"].items()):
        linhas.append(f"  Coluna {coluna} {COLUNAS_CARTELA[coluna]} -> {qtd} ocorrências")

    sugestoes = gerar_sugestoes(resultado, quantidades=quantidades, pontuacao_nn=pontuacao_nn)

    linhas.append("\n" + "=" * 70)
    qtds_str = ", ".join(str(q) for q in quantidades)
    linhas.append(f"SUGESTÕES PARA O PRÓXIMO CONCURSO ({qtds_str} números)")
    linhas.append("=" * 70)
    for nome, por_quantidade in sugestoes.items():
        linhas.append(f"\n{nome}:")
        for q, jogo in por_quantidade.items():
            n_pares = sum(1 for d in jogo if d % 2 == 0)
            linhas.append(
                f"  {q} números: {' - '.join(f'{d:02d}' for d in jogo)}  "
                f"(soma {sum(jogo)}, {n_pares}p/{q - n_pares}i)"
            )

    if pontuacao_nn is None:
        linhas.append(
            "\n(Categoria 'Rede neural (MLP)' não incluída - scikit-learn "
            "indisponível ou histórico insuficiente para treiná-la nesta execução.)"
        )

    linhas.append(
        "\nCritério da estratégia combinada: uma pontuação por dezena que mistura\n"
        "frequência histórica, tendência recente e atraso; o núcleo de 15 dezenas\n"
        "é ajustado para manter a soma dentro da faixa mais comum no histórico\n"
        "(média +/- 1 desvio padrão), e as dezenas extras (16ª, 17ª...) são as\n"
        "próximas melhor colocadas no mesmo ranking.\n"
        "Critério da rede neural: um MLP (perceptron multicamadas) treinado para\n"
        "prever, a partir da frequência de cada dezena na janela de concursos mais\n"
        "recente, a probabilidade de cada dezena sair no próximo concurso; as N\n"
        "dezenas de maior probabilidade prevista formam a sugestão."
    )

    linhas.append("\n" + "=" * 70)
    linhas.append("AVISO IMPORTANTE")
    linhas.append("=" * 70)
    linhas.append(
        "A Lotofácil é um sorteio aleatório: cada concurso é um evento independente\n"
        "e o resultado de concursos passados NÃO influencia o próximo sorteio.\n"
        "Nenhum método estatístico garante ou comprovadamente aumenta a chance de\n"
        "acerto. Esta análise é uma ferramenta exploratória/educacional para\n"
        "observar padrões históricos (frequência, atraso, soma, paridade). Jogue\n"
        "com responsabilidade."
    )

    return "\n".join(linhas)



In [ ]:
# paleta sequencial (uma única cor, do claro ao escuro) usada nos gráficos abaixo
AZUL_SEQUENCIAL = LinearSegmentedColormap.from_list(
    "azul_sequencial", ["#cde2fb", "#6da7ec", "#2a78d6", "#104281"]
)
COR_BARRA_UNICA = "#2a78d6"
COR_GRADE = "#e1e0d9"
COR_EIXO = "#c3c2b7"
COR_TEXTO_SECUNDARIO = "#52514e"

plt.rcParams.update({
    "figure.facecolor": "#fcfcfb",
    "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": COR_EIXO,
    "axes.grid": True,
    "grid.color": COR_GRADE,
    "grid.linewidth": 0.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "text.color": "#0b0b0b",
    "axes.labelcolor": COR_TEXTO_SECUNDARIO,
    "xtick.color": COR_TEXTO_SECUNDARIO,
    "ytick.color": COR_TEXTO_SECUNDARIO,
})

print("Funções e configuração prontas.")
if not SKLEARN_DISPONIVEL:
    print("Aviso: scikit-learn não encontrado - a categoria 'Rede neural' ficará indisponível.")
    print("Instale com: pip install scikit-learn")


## 1. Baixar/atualizar o histórico

Na primeira execução baixa todos os concursos (pode levar alguns minutos); nas próximas, baixa só os concursos novos. Isso sempre mantém o cache local completo — o corte de quantos concursos *usar na análise* é feito no próximo passo, sem precisar baixar de novo.

In [ ]:
historico = atualizar_historico()  # use atualizar_historico(forcar=True) para reforçar o download completo
lista_completa = historico_ordenado(historico)

print(f"Total de concursos em cache: {len(lista_completa)}")
print(f"Do concurso {lista_completa[0][0]} ao {lista_completa[-1][0]} (último sorteio em {lista_completa[-1][1]})")


## 2. Parâmetros da análise

- **`CONCURSOS_A_CONSIDERAR`**: quantos concursos *anteriores* (mais recentes) usar na análise e no treino da rede neural. Coloque, por exemplo, `100` para considerar só os últimos 100 concursos, ou `None` para usar todo o histórico baixado.
- **`JANELA_RECENTE`**: quantos concursos recentes entram na análise de tendência (frequência recente x histórica).
- **`JANELA_NN`**: quantos concursos a rede neural olha "para trás" para prever o próximo.
- **`QUANTIDADES`**: os tamanhos de aposta mostrados para cada estratégia (padrão 15, 16 e 17 números).

In [ ]:
CONCURSOS_A_CONSIDERAR = 100  # ex.: 100 -> usa só os 100 concursos mais recentes; None -> usa todo o histórico
JANELA_RECENTE = 25
JANELA_NN = 10
QUANTIDADES = (15, 16, 17)  # a Lotofácil aceita apostas de 15 a 20 números

lista = lista_completa[-CONCURSOS_A_CONSIDERAR:] if CONCURSOS_A_CONSIDERAR else lista_completa

print(f"Concursos usados nesta análise: {len(lista)} (de {lista[0][0]} a {lista[-1][0]})")

resultado = analisar(lista, janela_recente=JANELA_RECENTE)
resultado["total_concursos"], resultado["soma_media"], resultado["soma_desvio"]


## 3. Números mais e menos frequentes

In [ ]:
df_freq = pd.DataFrame({
    "dezena": list(range(1, 26)),
    "frequencia_total": [resultado["freq_total"].get(d, 0) for d in range(1, 26)],
    "frequencia_recente": [resultado["freq_recente"].get(d, 0) for d in range(1, 26)],
    "atraso": [resultado["atraso"].get(d, 0) for d in range(1, 26)],
})
df_freq["pct_total"] = (100 * df_freq["frequencia_total"] / resultado["total_concursos"]).round(1)

df_freq.sort_values("frequencia_total", ascending=False).head(10).reset_index(drop=True)


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ordem = df_freq.sort_values("dezena")
cores = AZUL_SEQUENCIAL(
    (ordem["frequencia_total"] - ordem["frequencia_total"].min())
    / (ordem["frequencia_total"].max() - ordem["frequencia_total"].min() + 1e-9)
)
ax.bar(ordem["dezena"], ordem["frequencia_total"], color=cores)
ax.set_xticks(range(1, 26))
ax.set_xlabel("Dezena")
ax.set_ylabel("Vezes sorteada")
ax.set_title("Frequência histórica de cada dezena")
plt.tight_layout()
plt.show()


## 4. Atraso — dezenas que não saem há mais tempo

In [ ]:
df_freq.sort_values("atraso", ascending=False).head(10).reset_index(drop=True)


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ordem_atraso = df_freq.sort_values("atraso", ascending=False)
ax.bar(ordem_atraso["dezena"].astype(str), ordem_atraso["atraso"], color=COR_BARRA_UNICA)
ax.set_xlabel("Dezena")
ax.set_ylabel("Concursos sem sair")
ax.set_title("Atraso de cada dezena (ordenado do mais ao menos atrasado)")
plt.tight_layout()
plt.show()


## 5. Pares de números que mais saem juntos

In [ ]:
df_pares = pd.DataFrame(
    [(a, b, qtd) for (a, b), qtd in resultado["pares"].most_common(15)],
    columns=["dezena_a", "dezena_b", "vezes_juntas"],
)
df_pares


## 6. Paridade e soma das 15 dezenas sorteadas

In [ ]:
df_paridade = pd.DataFrame(
    sorted(resultado["contagem_pares_impares"].items()),
    columns=["quantidade_de_pares", "concursos"],
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(df_paridade["quantidade_de_pares"].astype(str), df_paridade["concursos"], color=COR_BARRA_UNICA)
ax.set_xlabel("Números pares no jogo (de 15)")
ax.set_ylabel("Concursos")
ax.set_title("Distribuição de pares x ímpares por concurso")
plt.tight_layout()
plt.show()

print(f"Soma média histórica: {resultado['soma_media']:.1f}  |  Desvio padrão: {resultado['soma_desvio']:.1f}")
print(f"Repetição média em relação ao concurso anterior: {resultado['repeticao_media']:.1f} dezenas")


## 7. Distribuição por coluna da cartela oficial

In [ ]:
df_coluna = pd.DataFrame(
    sorted(resultado["freq_coluna"].items()),
    columns=["coluna", "ocorrencias"],
)
df_coluna["dezenas"] = df_coluna["coluna"].map(lambda c: COLUNAS_CARTELA[c])
df_coluna


## 8. Treinar a rede neural

Treina o MLP com os concursos selecionados em `CONCURSOS_A_CONSIDERAR` e obtém a probabilidade prevista de cada dezena sair no próximo concurso. Repita o lembrete: isso **não** significa que a rede "aprendeu a prever" a Lotofácil — é só mais uma pontuação por dezena, tão sujeita ao acaso quanto as outras.

In [ ]:
pontuacao_nn = None
if SKLEARN_DISPONIVEL:
    try:
        pontuacao_nn = treinar_rede_neural(lista, janela=JANELA_NN)
        df_nn = pd.DataFrame(
            sorted(pontuacao_nn.items(), key=lambda kv: kv[1], reverse=True),
            columns=["dezena", "probabilidade_prevista"],
        )
        display(df_nn.head(10))
    except Exception as exc:
        print(f"Não foi possível treinar a rede neural: {exc}")
else:
    print("scikit-learn não está instalado - pule esta célula ou rode: pip install scikit-learn")


## 9. Sugestões para o próximo concurso (15, 16 e 17 números)

Para cada estratégia, uma sugestão para cada tamanho de aposta:

- **Combinada**: mistura frequência histórica + tendência recente + atraso; o núcleo de 15 dezenas é ajustado para manter a soma na faixa mais comum do histórico, e as dezenas extras (16ª, 17ª) são as próximas melhor colocadas no mesmo ranking.
- **Mais quentes / Tendência recente / Números atrasados**: rankings simples por um único critério.
- **Rede neural (MLP)**: as dezenas com maior probabilidade prevista pelo modelo treinado acima.

In [ ]:
sugestoes = gerar_sugestoes(resultado, quantidades=QUANTIDADES, pontuacao_nn=pontuacao_nn)

linhas_tabela = []
for estrategia, por_quantidade in sugestoes.items():
    for q, jogo in por_quantidade.items():
        n_pares = sum(1 for d in jogo if d % 2 == 0)
        linhas_tabela.append({
            "estrategia": estrategia,
            "quantidade": q,
            "dezenas": " - ".join(f"{d:02d}" for d in jogo),
            "soma": sum(jogo),
            "pares": n_pares,
            "impares": q - n_pares,
        })

df_sugestoes = pd.DataFrame(linhas_tabela)
pd.set_option("display.max_colwidth", None)
df_sugestoes


## 10. Relatório completo (texto)

In [ ]:
relatorio = montar_relatorio(resultado, lista, quantidades=QUANTIDADES, pontuacao_nn=pontuacao_nn)
print(relatorio)

# opcional: salvar em arquivo
# Path("relatorio_lotofacil.txt").write_text(relatorio, encoding="utf-8")


---

**Aviso:** a Lotofácil é um sorteio aleatório. O resultado de concursos passados não influencia o próximo sorteio — e isso vale também para a rede neural, que não tem padrão real para aprender em um sorteio uniforme. Nenhuma das estratégias acima garante ou comprovadamente aumenta a chance de acerto. Use esta ferramenta apenas para fins exploratórios/educacionais e jogue com responsabilidade.